# 04 — LLM Persona Interpretation

Use **Claude API** (`claude-sonnet-4-6`) to read each cluster's aggregated feature profile
and label it with a meaningful customer **Persona**.

**LLM is called once per cluster** — it reads the cluster statistics (not individual customers)
and returns a persona name + description.

**Clarity gate:**
- If LLM returns confident, distinct personas → proceed to `05_classifier.ipynb`
- If LLM says clusters are ambiguous/overlapping → go back to `03_clustering.ipynb`,
  increase `K_MIN` (e.g. try more clusters), re-run, then re-run this notebook

Output: `outputs/personas.json`

In [ ]:
import yaml

# ── Load pipeline configuration ───────────────────────────────────────────────
with open('config.yaml') as f:
    _cfg = yaml.safe_load(f)

PERSONA_TONE = str(_cfg.get('persona_tone', 'easy')).lower()

TONE_INSTRUCTIONS = {
    'easy':
        'Use plain, everyday language. Avoid jargon. Use simple analogies a '
        'non-technical person would understand. Keep sentences short and friendly.',
    'professional':
        'Use formal business language. Frame insights as actionable recommendations '
        'suitable for executive presentations. Be concise and authoritative.',
    'data-driven':
        'Emphasise specific numbers and statistical ratios throughout. Quantify '
        'everything you can. Use technical language and cite exact metric values.',
    'creative':
        'Use vivid metaphors, storytelling, and imaginative analogies. Make each '
        'persona feel like a character with a story and personality.',
}.get(PERSONA_TONE, 'Use plain, everyday language. Avoid jargon.')

print(f'Persona tone : "{PERSONA_TONE}"')
print(f'Instructions : {TONE_INSTRUCTIONS}')

In [1]:
import json, os
import anthropic

# Set API key via environment variable:  export ANTHROPIC_API_KEY="sk-ant-..."
# Or uncomment and set directly:
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

client = anthropic.Anthropic()

with open('outputs/cluster_profiles.json') as f:
    profiles = json.load(f)

print(f'Loaded {len(profiles)} cluster profiles')
print('Cluster sizes:', {cid: p['n_customers'] for cid, p in profiles.items()})

Loaded 4 cluster profiles
Cluster sizes: {'0': 239, '1': 326, '2': 361, '3': 22}


## Build Prompt — Per Cluster

The prompt feeds the LLM the **per-category time-windowed stats** so it can reason about
which categories this cluster dominates, how loyally they shop there, and how much they spend.

## Call Claude API — All Clusters in One Call

Sending all clusters together so the LLM can compare and assign **distinct** personas.

In [ ]:
def build_all_clusters_prompt(profiles: dict, tone_instructions: str) -> str:
    """Build a single prompt with ALL clusters so the LLM can compare and name distinctly."""

    blocks = []
    for cid, profile in profiles.items():
        n  = profile['n_customers']
        ov = profile['overall']
        cat_stats = profile['category_stats']
        algo = profile.get('algorithm_detail', 'clustering')

        cat_lines = []
        for cat, s in list(cat_stats.items()):
            if s['n_txn_12m'] > 0:
                flag = ' ◀' if s['rel_n_txn'] >= 1.4 else ''
                cat_lines.append(
                    f"    {cat:<20} txns/yr={s['n_txn_12m']:>5}  "
                    f"spend/yr=${s['total_amt_12m']:>8}  "
                    f"avg/txn=${s['avg_spend_12m']:>7}  "
                    f"consec={s['consec_months']:>4}  "
                    f"vs_avg={s['rel_n_txn']:>4}x{flag}"
                )

        blocks.append(
            f"--- CLUSTER {cid} ({n} customers)  [{algo}] ---\n" +
            '\n'.join(cat_lines) +
            f"\n  Overall: avg_txn=${ov['avg_txn_amt']}  "
            f"total_spend=${ov['total_spend']}  "
            f"txns_per_yr={ov['total_txn_count']}  "
            f"pct_high_value={ov['pct_high_value']}%  "
            f"avg_days_between={ov['avg_days_between_txn']}"
        )

    all_blocks = '\n\n'.join(blocks)
    n_clusters = len(profiles)

    return f"""You are a consumer behavior analyst. Below are {n_clusters} customer clusters from bank transaction data.
Each row shows a spending category with: annual transactions, annual spend, avg per transaction, consecutive loyal months, and vs_avg (1.0 = typical customer, higher = above average).
Categories marked ◀ are where the cluster is significantly above average.

{all_blocks}

Your task: assign a UNIQUE persona to EACH cluster. Since some clusters may look similar, focus on their subtle differences:
- Frequency differences (txns/yr, avg_days_between)
- Spend intensity (avg/txn, total_spend)
- Loyalty patterns (consec months)
- Which specific categories they index HIGHEST on vs_avg

TONE REQUIREMENT: {tone_instructions}
Apply this tone to persona names, taglines, descriptions, and traits.

Rules:
- Every persona name must be DIFFERENT — no duplicates allowed
- Focus on what makes each cluster distinct from the others
- Persona names should reflect the UNIQUE character of that cluster

Return ONLY a valid JSON object (no markdown, no extra text) with this structure:
{{
  "0": {{
    "name": "...",
    "tagline": "...",
    "description": "...",
    "dominant_categories": ["cat1", "cat2", "cat3"],
    "traits": ["trait1", "trait2", "trait3", "trait4", "trait5"],
    "confidence": <1-10>
  }},
  "1": {{ ... }},
  ...
}}"""


# Single API call for all clusters
print(f'Calling Claude with all {len(profiles)} clusters (tone: "{PERSONA_TONE}")...')
prompt = build_all_clusters_prompt(profiles, TONE_INSTRUCTIONS)

response = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': prompt}]
)

raw = response.content[0].text.strip()

# Strip markdown fences if present
if '```' in raw:
    parts = raw.split('```')
    for part in parts:
        p = part.strip()
        if p.startswith('json'): p = p[4:].strip()
        if p.startswith('{'): raw = p; break

try:
    personas = json.loads(raw)
    print('Parsed successfully.\n')
    for cid, p in personas.items():
        print(f'  Cluster {cid}: "{p["name"]}"  conf={p["confidence"]}/10')
except json.JSONDecodeError as e:
    print(f'JSON parse error: {e}')
    print('Raw:', raw[:500])
    personas = {}

## Save Results

In [3]:
combined = {
    cid: {'cluster_stats': profiles[cid], 'persona': personas.get(cid, {})}
    for cid in profiles
}

with open('outputs/personas.json', 'w') as f:
    json.dump(combined, f, indent=2)

print('Saved outputs/personas.json')

Saved outputs/personas.json


## Clarity Gate

Assess whether the personas are distinct enough to proceed to the classifier.

**Rule:** if LLM confidence < 6 on average, or any two personas share the same
dominant categories → go back to `03_clustering.ipynb` and increase k.

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

features = pd.read_parquet('data/processed/customer_features_clustered.parquet')
CATEGORIES = ['entertainment','food_dining','gas_transport','grocery_net','grocery_pos',
              'health_fitness','home','kids_pets','misc_net','misc_pos',
              'personal_care','shopping_net','shopping_pos','travel']
WINDOWS = [6, 12]
LOG_COLS = (
    [f'n_txn_{cat}_{w}m'     for cat in CATEGORIES for w in WINDOWS] +
    [f'amt_{cat}_{w}m'       for cat in CATEGORIES for w in WINDOWS] +
    [f'avg_spend_{cat}_{w}m' for cat in CATEGORIES for w in WINDOWS] +
    ['total_txn_count', 'total_spend', 'avg_txn_amt', 'std_txn_amt',
     'max_txn_amt', 'n_unique_merchants', 'avg_days_between_txn']
)
X = features.drop(columns=['cluster']).copy()
for col in LOG_COLS:
    if col in X.columns:
        X[col] = np.log1p(X[col])
X_scaled = StandardScaler().fit_transform(X)

sil = silhouette_score(X_scaled, features['cluster'])

print('=== Clarity Gate Assessment ===')
print(f'Silhouette score   : {sil:.4f}  (threshold ≥ 0.15)')
print()

confidences, names = [], []
for cid, data in combined.items():
    p = data['persona']
    if 'name' in p:
        conf = p.get('confidence', 0)
        confidences.append(conf)
        names.append(p['name'])
        print(f'  Cluster {cid}: "{p["name"]}"  confidence={conf}/10')

avg_conf = np.mean(confidences) if confidences else 0
names_unique = len(names) == len(set(names))

print(f'\nAvg LLM confidence : {avg_conf:.1f}/10  (threshold ≥ 6.0)')
print(f'Persona names unique: {names_unique}')

PROCEED = sil >= 0.15 and avg_conf >= 6.0 and names_unique
if PROCEED:
    print('\n✅ CLARITY GATE PASSED → run 05_classifier.ipynb')
else:
    print('\n⚠️  CLARITY GATE FAILED:')
    if sil < 0.15:       print('   • Silhouette too low → adjust k or features in 03_clustering.ipynb')
    if avg_conf < 6.0:   print('   • LLM confidence low → increase k')
    if not names_unique: print('   • Duplicate persona names → increase k for finer separation')

=== Clarity Gate Assessment ===
Silhouette score   : 0.1836  (threshold ≥ 0.15)

  Cluster 0: "Dormant Dabblers"  confidence=8/10
  Cluster 1: "Hyperactive Omnivores"  confidence=9/10
  Cluster 2: "Steady Commuter Mainstays"  confidence=7/10
  Cluster 3: "Rare Big-Ticket Splurgers"  confidence=9/10

Avg LLM confidence : 8.2/10  (threshold ≥ 6.0)
Persona names unique: True

✅ CLARITY GATE PASSED → run 05_classifier.ipynb


## Persona Summary

In [5]:
print('\n' + '='*65)
print('PERSONA SUMMARY')
print('='*65)
for cid, data in combined.items():
    p = data['persona']
    s = data['cluster_stats']
    if 'name' in p:
        print(f'\nCluster {cid}  ({s["n_customers"]} customers)')
        print(f'  Persona  : {p["name"]}')
        print(f'  Tagline  : {p["tagline"]}')
        print(f'  Dominant : {p["dominant_categories"]}')
        print(f'  Traits:')
        for t in p.get('traits', []):
            print(f'    • {t}')


PERSONA SUMMARY

Cluster 0  (239 customers)
  Persona  : Dormant Dabblers
  Tagline  : Present across every category, committed to none
  Dominant : ['grocery_pos', 'home', 'gas_transport']
  Traits:
    • Below-average spend across every single category — no standout area
    • Lowest transaction volume per year (476.8) of all active clusters
    • Longest average days between transactions (0.6), indicating infrequent card use
    • Moderate loyalty months suggest they haven't left but aren't deeply engaged
    • Likely use this card as a secondary or backup payment method

Cluster 1  (326 customers)
  Persona  : Hyperactive Omnivores
  Tagline  : All-in, all the time, on everything
  Dominant : ['shopping_pos', 'grocery_pos', 'food_dining']
  Traits:
    • Above average in ALL 14 spending categories simultaneously — unique among all clusters
    • Highest annual transaction count by far (2,230/yr), near-zero days between purchases
    • Maximum consecutive loyalty months (18.0) acro